# 3D coasts: useful depth, dangerous certainty

**Atlas source reading copy · browser-network**

[Course guide](../index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-Sea-Level-Rise/blob/66065673b47f9bc5c0f6e5009ac1c924a9f5200c/content/notebooks/07_three_dimensions.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability.

**Provenance:** `content/notebooks/07_three_dimensions.ipynb` at `66065673b47f9bc5c0f6e5009ac1c924a9f5200c`; original SHA-256 `3926f9b68d39aa1720f274e787b89bebdb1b546d59c4ab957a3549b51d5c21ea`. Source-specific license/notices retained under notices; data and media have separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# 3D coasts: useful depth, dangerous certainty

**Time:** 40 minutes · **Prerequisites:** basic Python; run cells from top to bottom.

Build a transparent terrain-and-water scene; distinguish elevation from geographic stems; explore sensitivity without implying hydrodynamic realism.

[Run this notebook in JupyterLite](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/lite/lab/index.html?path=notebooks/07_three_dimensions.ipynb) · [Full-screen dashboard](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/dashboard/) · [Official poster](https://plus.iasociety.org/sites/default/files/2024-09/e-poster_755.pdf)

Each notebook runs independently. Use **Run → Run All Cells**; the first package download can take a minute. Save a downloaded copy to keep your work. The read-only book includes executed examples.

In [ ]:
# First run downloads Python packages in JupyterLite; subsequent runs use browser caches.
import sys
if sys.platform == 'emscripten':
    import piplite
    await piplite.install(['numpy', 'pandas', 'matplotlib', 'plotly', 'ipywidgets'])
from pathlib import Path
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, HTML
from coastlab import (DATA, SCENARIOS, YEARS, COLORS, load_data, select, cards,
                      toy_slr, toy_probability, cumulative_probability,
                      haversine_km, show, dashboard, land_traces)
facilities, exposure = load_data()
published = pd.read_csv(DATA / 'poster_aggregates.csv')
print('Ready: public poster aggregates + clearly labeled fictional facility fixtures.')

## What does height represent?

A three-dimensional display can encode physical elevation, a statistical value or time. Those meanings are not interchangeable. A map with stems whose height is the percentage of exposed facilities is a geographic chart, not a digital elevation model. A space–time cube uses its vertical axis for time.

The fictional terrain below is an analytic surface over a 4 × 4 km square. It is independent of the city fixtures and has no real coordinate reference system. The water plane is a threshold illustration. It does not simulate water flowing around barriers, wave overtopping, tides, drainage or river flooding.

In [ ]:
x=np.linspace(0,4,51);y=np.linspace(0,4,51)
X,Y=np.meshgrid(x,y)
terrain=.15+.48*X+.16*np.sin(2*X+Y)+.08*np.cos(3*Y)
water=.7+toy_slr('RCP 8.5',2100)
fig=go.Figure([go.Surface(x=x,y=y,z=terrain,colorscale='Earth',showscale=False,name='Invented terrain'),
              go.Surface(x=x,y=y,z=np.full_like(terrain,water),opacity=.55,colorscale=[[0,'#229bd1'],[1,'#229bd1']],showscale=False)])
fig.update_layout(title='Independent fictional coast — exaggerated vertical scale',
    scene=dict(xaxis_title='x (km)',yaxis_title='y (km)',zaxis_title='Toy height (m)',aspectratio=dict(x=1,y=1,z=.6)))
show(fig)

In [ ]:
levels=np.arange(.25,2.01,.25)
sensitivity=pd.DataFrame({'water_m':levels,'cells_below_plane_percent':[100*np.mean(terrain<=level) for level in levels]})
display(sensitivity)
print('This is the share of sampled grid cells below a plane, not a validated inundated area.')

In [ ]:
dashboard(view='terrain',dataset='synthetic')

## Lab: make the missing physics visible

Add an elevated ridge to the surface. Compare a simple below-water mask to a connected-ocean mask using a flood-fill starting from the left edge. The second mask should exclude isolated inland depressions until a connected path exists. Neither method supplies a full hydrodynamic model.

**Worked approach:** Use a queue; initialize ocean-edge cells below the water level; visit adjacent below-water cells until no new cells remain. Keep cell resolution fixed. Quantify the difference between all low cells and connected low cells.

**Deliverable:** Two labelled masks and a 3D scene with an explicit vertical-exaggeration note. List the data needed for a real analysis: surveyed heights, datum transformation, local coastal water levels, defences, connectivity and validation events.

Source context: [CoastalDEM v2.1](https://www.climatecentral.org/coastaldem-v2.1). No CoastalDEM raster is distributed or simulated by this surface.